In [2]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme.git

Cloning into 'Dual_watermarking_Scheme'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 163 (delta 0), reused 5 (delta 0), pack-reused 151 (from 1)
Receiving objects: 100% (163/163), 42.77 MiB | 9.44 MiB/s, done.
Resolving deltas: 100% (59/59), done.


In [3]:
%cd Dual_watermarking_Scheme/

/content/Dual_watermarking_Scheme


In [4]:
from huggingface_hub import login
login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [6]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [7]:
import torch
import pandas as pd
from transformers import LogitsProcessor

In [8]:
from src.watermark.first_layer import load_topic_greenlists,build_topic_matrix
from src.utils.key_manager import derive_set

In [9]:
"""
Applies both watermark boosts to the next-token logits at every step.
boosted_logit = base_logit + w1 * delta_public + w2 * delta_private
"""

'\nApplies both watermark boosts to the next-token logits at every step.\nboosted_logit = base_logit + w1 * delta_public + w2 * delta_private\n'

In [10]:
class DualLayerWatermarkProcessor(LogitsProcessor):
    """
    Applies both watermark boosts to the next-token logits at every step.
    boosted_logit = base_logit + w1 * delta_public + w2 * delta_private
    """
    def __init__(self, topic_green_ids, key, vocab_size, green_fraction=0.5,
                 delta_public=2.0, delta_private=0.7, w1=1.0, w2=1.0, prev_token_size=5):
        self.topic_green_ids = torch.tensor(topic_green_ids, dtype=torch.long)
        self.key = key
        self.vocab_size = vocab_size
        self.green_fraction = green_fraction
        self.delta_public = delta_public
        self.delta_priv = delta_private
        self.w1 = w1
        self.w2 = w2
        self.prev_token_size = prev_token_size

    def __call__(self, input_ids, scores):
        if self.w1 != 0.0:
            topic_ids = self.topic_green_ids.to(scores.device)
            scores[..., topic_ids] += self.w1 * self.delta_public
        if self.w2 != 0.0:
            batch_size = input_ids.shape[0]
            for b in range(batch_size):
                history = input_ids[b].tolist()
                preferred_set = derive_set(
                    vocab_size=self.vocab_size,
                    green_fraction=self.green_fraction,
                    key=self.key,
                    prev_tokens_size=self.prev_token_size,
                    prev_tokens=history,
                )
                preferred = torch.tensor(
                    list(preferred_set), dtype=torch.long, device=scores.device
                )
                scores[b, preferred] += self.w2 * self.delta_priv
        return scores

In [11]:
class DualWaterMarking:
    """
    This is class for end to end dual watermarking . It will do the following:-
    1. First, it will extract topic from prompt (same as layer 1).
    2. build a DualLayerWatermarkProcessor for layer 1 + layer 2 functionaility. 
    3. generate plain vs dual watermarked text for comparison
    """

    def __init__(self,model,tokenizer,key,greenlist_csv="topic_greenlists.csv",green_fraction=0.5,delta_public=2.0,
                 delta_private=0.7,w1 = 1.0,w2=1.0,prev_token_size=5,temperature=1.0,top_p=0.9,max_new_tokens=50):

        self.model = model.eval()
        self.tokenizer = tokenizer
        self.key = key
        self.vocab_size = model.get_input_embeddings().weight.shape[0]
 
        self.green_fraction = green_fraction
        self.delta_public = delta_public
        self.delta_priv = delta_private
        self.w1 = w1
        self.w2 = w2
        self.prev_token_size = prev_token_size
 
        self.temperature = temperature
        self.top_p = top_p
        self.max_new_tokens = max_new_tokens
 
        self.greenlists = load_topic_greenlists(greenlist_csv)
        self.topics = sorted(self.greenlists)
        self.topic_matrix = build_topic_matrix(model, tokenizer, self.topics)

    @torch.no_grad()
    def extract_topic(self,prompt):
        """Mean-pool prompt token embeddings, cosine vs each topic vector."""
        ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        vec = self.model.get_input_embeddings().weight[ids].mean(dim=0)
        vec = vec / vec.norm()
        scores = self.topic_matrix @ vec
        ranked = sorted(zip(self.topics, scores.tolist()), key=lambda x: -x[1])
        return ranked[0][0], ranked

    def _make_processor(self,topic,w1,w2):

        return DualLayerWatermarkProcessor(
            topic_green_ids=self.greenlists[topic],
            key=self.key,
            vocab_size=self.vocab_size,
            green_fraction=self.green_fraction,
            delta_public=self.delta_public,
            delta_private=self.delta_priv,
            w1=w1,
            w2=w2,
            prev_token_size=self.prev_token_size,
        )

    def _generate(self, inputs, processors=None):
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                temperature=self.temperature,
                top_p=self.top_p,
                logits_processor=processors,
            )
        return self.tokenizer.decode(out[0], skip_special_tokens=True)

    def watermark(self, prompts, include_single_layers=True):
        """
        Generate plain vs dual-watermarked output for each prompt.
        If include_single_layers=True, also generates Layer-1-only (w2=0) and
        Layer-2-only (w1=0) outputs - useful for Phase 7 ablation/evaluation.
        Returns a DataFrame.
        """
        rows = []
        for i, prompt in enumerate(prompts):
            topic, ranked = self.extract_topic(prompt)
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
 
            plain = self._generate(inputs)
            dual_output = self._generate(
                inputs, processors=[self._make_processor(topic, self.w1, self.w2)]
            )
 
            row = {
                "id": i,
                "prompt": prompt,
                "topic": topic,
                "topic_score": round(ranked[0][1], 4),
                "plain_output": plain,
                "dual_watermarked_output": dual_output,
            }
 
            if include_single_layers:
                row["layer1_only_output"] = self._generate(
                    inputs, processors=[self._make_processor(topic, 1.0, 0.0)]
                )
                row["layer2_only_output"] = self._generate(
                    inputs, processors=[self._make_processor(topic, 0.0, 1.0)]
                )
 
            rows.append(row)
 
        return pd.DataFrame(rows)

TESTING THE DUAL MARK THING

In [12]:
from src.utils.loadConfig import load_config
config = load_config("secondLayer")
config

{'MODEL_NAME': 'facebook/opt-2.7b',
 'GREEN_FRACTION': 0.5,
 'PREV_TOKEN_SIZE': 5,
 'DETECTION_THRESHOLD': 0.6,
 'P_VALUE_THRESHOLD': 0.05}

In [13]:
from src.utils.model import load_model
model,tokenizer,vocab_size = load_model(config["MODEL_NAME"])
model.to(device)

from src.utils.key_manager import generate_key
key = generate_key()
print(f"secret key is generated: {key}")

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.30GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model and tokenizer of facebook/opt-2.7b loaded with vocab_size 50265
secret key is generated: 67e5ee955d7dc262cefaa0f57483e06dc24047d591ada32be6b7c9e460a5fd70


In [14]:
import random

In [ ]:
topic_csv_path = "topic_greenlists.csv"
# TOPICS = ["science", "sports", "cooking"] 
# GREEN_TOKENS_PER_TOPIC = 300
 
_rng = random.Random(42)
rows = [
    {"topic": t, "token_id": tid}
    for t in TOPICS
    for tid in _rng.sample(range(vocab_size), GREEN_TOKENS_PER_TOPIC)
]
pd.DataFrame(rows).to_csv(topic_csv_path, index=False)
print(f"Wrote sample greenlists for topics {TOPICS} -> {topic_csv_path}")
 
dwm = DualWaterMarking(
    model=model,
    tokenizer=tokenizer,
    key=key,
    greenlist_csv=topic_csv_path,
    green_fraction=config["GREEN_FRACTION"],
    delta_public=2.0,
    delta_private=0.7,        # <-- was delta_priv, now matches your class's param name
    w1=1.0,
    w2=1.0,
    prev_token_size=config["PREV_TOKEN_SIZE"],
    max_new_tokens=40,
)
 
sample_prompts = [
    "The recent breakthrough in quantum computing suggests that",
    "During the final minutes of the championship game,",
    "To make the perfect homemade pizza dough, you need",
]
 
results_df = dwm.watermark(sample_prompts, include_single_layers=True)
 
pd.set_option("display.max_colwidth", None)
for _, r in results_df.iterrows():
    print("=" * 80)
    print(f"PROMPT   : {r['prompt']}")
    print(f"TOPIC    : {r['topic']}  (score={r['topic_score']})")
    print(f"PLAIN    : {r['plain_output']}")
    print(f"LAYER1   : {r['layer1_only_output']}")
    print(f"LAYER2   : {r['layer2_only_output']}")
    print(f"DUAL     : {r['dual_watermarked_output']}")
print("=" * 80)
 
results_df

Wrote sample greenlists for topics ['science', 'sports', 'cooking'] -> topic_greenlists.csv
PROMPT   : The recent breakthrough in quantum computing suggests that
TOPIC    : science  (score=0.1868)
PLAIN    : The recent breakthrough in quantum computing suggests that quantum computer will be very fast, but as the size of the quantum bits (qubits) increases, the cost of the qubits also increase dramatically. Thus, the current qubits are limited in
LAYER1   : The recent breakthrough in quantum computing suggests that it's time for the field to expand its horizons.

Researchers from the University of California, Berkeley, the University of Illinois at Urbana-Champaign, and the University of California
LAYER2   : The recent breakthrough in quantum computing suggests that we may be close to discovering the missing link in the evolution of life itself. Credit: Nick Gudgeon

As the human race continues to advance, it increasingly seems like we're heading for
DUAL     : The recent breakthrough 

,id,prompt,topic,topic_score,plain_output,dual_watermarked_output,layer1_only_output,layer2_only_output
0,0,The recent breakthrough in quantum computing suggests that,science,0.1868,"The recent breakthrough in quantum computing suggests that quantum computer will be very fast, but as the size of the quantum bits (qubits) increases, the cost of the qubits also increase dramatically. Thus, the current qubits are limited in","The recent breakthrough in quantum computing suggests that there could be a way to harness the power of quantum dots to build more efficient computers, and the U.S. government is now looking for ways to help get it done.\n\nThe National","The recent breakthrough in quantum computing suggests that it's time for the field to expand its horizons.\n\nResearchers from the University of California, Berkeley, the University of Illinois at Urbana-Champaign, and the University of California","The recent breakthrough in quantum computing suggests that we may be close to discovering the missing link in the evolution of life itself. Credit: Nick Gudgeon\n\nAs the human race continues to advance, it increasingly seems like we're heading for"
1,1,"During the final minutes of the championship game,",sports,0.2150,"During the final minutes of the championship game, the crowd at TD Garden was so quiet, you could hear a pin drop.\n\nIt was one of the quietest moments in recent playoff history. There were no hollering fans, no","During the final minutes of the championship game, with just over a minute left in regulation, the Saints were awarded a free kick. The call was that the ball had crossed the line from behind the goal. The referees looked at the replay of the","During the final minutes of the championship game, when the teams were playing for second place in the tournament, I caught myself wondering, “Who does that guy look like?”\n\nNot to offend anyone, but I didn’","During the final minutes of the championship game, the Warriors are clinging to a two-point lead and the Cavaliers have a chance to extend the lead even further. LeBron James draws up a pass to Kyrie Irving, who is lined up against K"
2,2,"To make the perfect homemade pizza dough, you need",sports,0.2041,"To make the perfect homemade pizza dough, you need the right ingredients, plus a little time and patience.\n\nThe secret ingredient in a great pizza is not the sauce or the crust but rather your dough. That's because dough is key for creating","To make the perfect homemade pizza dough, you need the following ingredients:\n\n• Olive oil • Fresh garlic • Salt and pepper • Flour • Baking soda • Egg • A splash of vinegar (or lemon juice) • A pinch of red","To make the perfect homemade pizza dough, you need the right ingredients, and this article contains your complete guide to making your own dough for your next pizza. Read on for our pizza dough tips!","To make the perfect homemade pizza dough, you need the best dough recipe out there!\n\nMany homemade pizzas start with a simple homemade pizza dough recipe and that’s the best homemade pizza dough recipe we’ve ever seen!\n"
